# Notebook 1: Data Exploration

Welcome!, The main goal for this notebook is to provide a deep understanding of the dataset, its characteristics, and any hidden patterns.

---

## Introduction

#### Data Description

This project analyzes radio continuum data from the MeerKAT International GHz Tiered Extragalactic Exploration ([MIGHTEE](https://idia.ac.za/mightee/)) survey, focusing on extragalactic radio emissions in the COSMOS field. MIGHTEE offers insights into galaxy evolution, given its depth and multiwavelength coverage. The COSMOS field, devoid of bright radio, UV, and X-ray sources, is ideal for deep astronomical surveys.

The MIGHTEE data includes deep fields like COSMOS, XMM-LSS, E-CDFS and ELAIS-S1, covering a 20 deg^2 total area at a central frequency of 1284 MHz. These fields were observed using MeerKAT's L-band receivers for about 1000 hours. They were chosen due to their extensive multiwavelength data from prior and potential future surveys.

The Mightee Cosmos Field: cred: [Hale et al.](https://arxiv.org/abs/2411.04958)

![My chart](./images/cosmos.jpg)

The **MIGHTEE-COSMOS multiwavelength catalogue** we will use here combines radio data with optical, near-to-far infrared, and X-ray measurements from various sources in the COSMOS field. The main radio catalogue was cross-matched with optical and near-infrared data from other studies, resulting in a matched set of 5,224 radio sources. Additionally, 572 of these sources were detected in X-ray observations. This dataset also incorporated Mid-Infrared (MIR) data, referencing specific IRAC flux densities. The catalogue further integrated far-infrared data from the Herschel Extra-galactic Legacy Project, identifying thousands of radio sources across different instruments.

#### Why Multiwavelength?
- Different wavelengths probe different physical processes and components of astronomical objects.
- For example, radio waves can reveal cold gas and magnetic fields, while X-rays can reveal hot gas and high-energy processes.
- By combining data from multiple wavelengths, astronomers can gain a more comprehensive understanding of the structure, composition, and dynamics of celestial objects and phenomena.

#### Example
Centaurus A in Different Wavelengths (Hubble, Spitzer, Chandra, VLA)

![My chart](images/centaurus_a.png)

Image_cred: https://science.nasa.gov/asset/webb/centaurus-a-in-different-wavelengths-hubble-spitzer-chandra-vla/

#### Why explore the data?

Data exploration serves as the foundational step in data analysis, aimed at understanding inherent structures, patterns, and potential anomalies within a dataset. It helps ascertain the quality of data, understand features' distributions, formulate hypotheses, guide preprocessing decisions, and select appropriate analytical tools and models. Efficient data exploration also aids in communicating initial insights to stakeholders. In this notebook, we will perform some of these critical tasks.

---

### Step 1: Import libraries needed for analysis

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import astropy.io.fits as fits 


---

### Step 2: Read data

In [ ]:
hdul = fits.open("data/updated_mightee_cat.fits")
mightee_data = pd.DataFrame(hdul[1].data)
hdul.close()

In [ ]:
mightee_data.shape

From above we can observe that we have 167 columns (which will be referred to as feateres) and 5223 rows. We can run the following command to see what are those feature names:

In [ ]:
list(mightee_data.columns)
# list(mightee_data.columns[:10]) # printing the first 10 columns
# These columns/ Features described in the following file @@ data/README_MIGHTEE.pdf. Please refer to the file for more details about the features.

### Step 3: Cleaning missing data

In [ ]:
# Now lets check if the dataframe has any missing values. This is important because missing values can affect the performance of machine learning models and may require imputation or removal of rows/columns with missing data.
# The output is True, which means there are missing values in the dataframe. This is not good news as it means we can not proceed with our analysis without worrying about missing data.
# It is also common in astronomy that data will contain negative values for flux such as -99. this is not an actual measurement and also represent missing values so it needs to be treated as missing data. 
# We will replace -99 with NaN (Not a Number) which is the standard way to represent missing values in pandas.

# converting -99 to NaN
mightee_data.replace(-99, np.nan, inplace=True)

# checking for missing values again after replacing -99 with NaN
print(mightee_data.isna().any().any()) 

The dataframe appears to contain missing values (And not sure how much so far), so our first step is to clean the data. As a starting point, the code block below removes all columns in the catalogue with more than 10% missing values. While this threshold is commonly used in practice, it is worth considering whether 10% is the most appropriate choice for this dataset. Are there alternative strategies for handling missing data that might preserve more information? We will explore these questions in the next task.


In [ ]:
# Check for missing values by column and remove columns with 90% or more missing data
missing_frac = mightee_data.isna().mean()
cols_to_drop = missing_frac[missing_frac >= 0.10].index.tolist()

if cols_to_drop:
    print(f"Deleting {len(cols_to_drop)} columns with >= 10% missing data:")
    for col in cols_to_drop:
        print(" -", col)
    mightee_data = mightee_data.drop(columns=cols_to_drop)
else:
    print("No columns with >= 10% missing data found.")


### Exercise 1 
- remove missing values from the remaining catalogue, ie remove any row that has a missing value.


In [ ]:
#### Answer to Exercise 1 here:

### Exercise 2 
- Uncomment and run the cell block below to remove the object coordinates from the data set and the ID colums

In [ ]:
# import re

# def is_coord_col(name: str) -> bool:
#     s = str(name).strip()
#     # Match standalone tokens 'ra'/'dec', occurrences separated by non-letters (e.g., 'ra_1a'),
#     # or as a suffix (e.g., 'RADIORA'). Avoid matching inside ordinary words like 'radius'.
#     pat = re.compile(r'(?i)(?:\b(?:ra|dec)\b|(?:^|[^A-Za-z])(?:ra|dec)(?:[^A-Za-z]|$)|(?:ra|dec)$)')
#     return bool(pat.search(s))


# def is_id_col(name: str) -> bool:
#     s = str(name).strip()
#     # Match exact token 'id', token separated by non-alphanumeric characters, or common ID suffixes like MATCHID.
#     return bool(re.search(r'(?i)(?:^id$|(?:^|[^A-Za-z0-9])id(?:$|[^A-Za-z0-9])|id$)', s))

# coord_cols = [col for col in mightee_data.columns if is_coord_col(col)]
# id_cols = [col for col in mightee_data.columns if is_id_col(col)]
# print("Coordinate columns found:", coord_cols)
# print("ID columns found:", id_cols)

# for col in coord_cols:
#     cleaned = (
#         mightee_data[col]
#         .astype(str)
#         .str.strip()
#         .replace({'': None, 'None': None, 'nan': None})
#     )
#     mightee_data[col] = pd.to_numeric(cleaned, errors='coerce')

# print("Cleaned coordinate columns:", coord_cols)

# cols_to_drop = list(dict.fromkeys(coord_cols + id_cols))
# mightee_data.drop(columns=cols_to_drop, inplace=True)
# print("Dropped columns:", cols_to_drop)

In [ ]:
#### Explain why we need to clean the coordinate columns and how we cleaned them here:

In [ ]:
### After some data cleaning lets see how many rows and columns we have left in the dataframe. 
print("Dataframe shape after cleaning:", mightee_data.shape)

### Step 4: Seperate the features used as input (X) from target (y)

In [ ]:
# Firstly, Since the MIGHTEE dataset contains multiple columns that indicate whether a source is an AGN, SFG, unclassified, or a probable SFG,
# we need to consolidate these into a single 'labels' column for our task. The function `create_label_column` will take care of this by applying the following logic:
# All you need to do is to run this cell

def create_label_column(df, y_features):
    """Takes a processed MIGHTEE DataFrame, creates a single 'labels'

    column (AGN or SFG), and drops the original four classification columns.

    Parameters:
    -----------
    df : pd.DataFrame
        The DataFrame containing the X features and the 4 target features.
    y_features : list
        A list of exactly 4 strings corresponding to:
        [AGN_col, SFG_col, probSFG_col, unclass_col]

    Returns:
    --------
    pd.DataFrame
        Cleaned DataFrame with a 'labels' column, minus unclassified and
    unnecessary target columns.
    """
    # 1. Create a shallow copy to prevent SettingWithCopyWarning
    processed_df = df.copy()

    # 2. Extract column names for clarity
    agn_col, sfg_col, probsfg_col, unclass_col = y_features

    # 3. Define the conditions and corresponding choices
    conditions = [
        processed_df[agn_col] == True,
        (processed_df[sfg_col] == True) | (processed_df[probsfg_col] == True),
    ]
    choices = ["AGN", "SFG"]

    # FIX: Changed default=np.nan to default='None' so all data types are strings
    processed_df["labels"] = np.select(conditions, choices, default="None")

    # 4. Print shapes before dropping
    print(
        "Shape of AGN: ", processed_df[processed_df["labels"] == "AGN"].shape
    )
    print(
        "Shape of SFG: ", processed_df[processed_df["labels"] == "SFG"].shape
    )
    print(
        "Shape of Unclassified/Dropped: ",
        processed_df[processed_df[unclass_col] == True].shape,
    )

    # 5. Drop rows that are marked as 'None' (the unclassified ones)
    processed_df = processed_df[processed_df["labels"] != "None"]

    # 6. Drop the original 4 target columns
    processed_df = processed_df.drop(columns=y_features)

    # 7. Clean up infinity values
    processed_df = processed_df.replace(-np.inf, np.nan)

    print("DONE PROCESSING")
    return processed_df

mightee_data = create_label_column(
    mightee_data,
    ["AGN", "SFG", "probSFG", "unclass"],
)

In [ ]:
# 1. Separate your target label from the features
X = mightee_data.drop(columns=["labels"])
y = mightee_data["labels"]

### Step 5 Data Types

#### Now lets check the data types
- The next step is to check the data types of the columns in the dataframe. 
- This is important because we want to make sure that all columns are numeric type (float) and not strings or bool. 
- If there are any `object type`, we will need to convert them to numeric type before we can use them for any analysis.

In [ ]:
print("Data types of columns:")
print(X.dtypes)
print("-" * 30)

# Check if ANY column contains object (string) or string data types
has_strings = any(
    pd.api.types.is_object_dtype(dtype)
    or pd.api.types.is_string_dtype(dtype)
    for dtype in X.dtypes
)

# 1. Identify which columns contain string/object data types
string_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

if string_cols:
    print(f"⚠️ Found string data in the following columns: {string_cols}")
    print("Dropping these columns to prepare for scaling...")

    # 2. Drop the string columns from X
    X_numeric = X.drop(columns=string_cols)
else:
    print("✅ No string columns found.")
    X_numeric = X.copy()

# 3. Check for any remaining missing values (NaNs) that might cause scaler to crash
if X_numeric.isna().any().any():
    print(
        "⚠️ Found some missing values (NaNs). Dropping rows with NaNs..."
    )  # scale can't take NaNs either
    X_numeric = X_numeric.dropna()

print("-" * 30)
print(
    f"🚀 Continuing with {X_numeric.shape[1]} purely numeric columns. Proceeding to scale..."
)

---

### Step 6: Descriptive Statistics:

**Descriptive statistics** provide a concise overview of the central tendencies, spread, and shape of the data distribution. By observing the mean, median, mode, and standard deviation, we can quickly grasp the characteristics of our dataset and identify potential anomalies or patterns.

In [ ]:
desc_stats = X_numeric.describe()

# Display the statistics
desc_stats

### Step 7: Data Normalization

From the descriptive statistics and exploratory visualizations, we observe that the features are measured on substantially different scales. Such scale differences can distort visual representations and obscure meaningful patterns in the data. To address this, we normalize the features to the range `[0, 1]`, ensuring that each feature contributes equally to the analysis and enabling more meaningful comparisons and visual interpretations.




In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
normalized_mightee = scaler.fit_transform(X_numeric)
normalized_mightee = pd.DataFrame(normalized_mightee, columns=X_numeric.columns)

In [ ]:
normalized_mightee.describe()

---

### Step 8: Freature selection

With 102 features, the dataset is highly dimensional, making visualization, interpretation, and clustering more challenging. To reduce complexity and eliminate redundancy, we can perform a correlation analysis to identify features that are strongly correlated with one another. Highly correlated features often contain overlapping information, so removing some of them can simplify the dataset while preserving most of its informative content. This results in a more compact feature set that is better suited for visualization and downstream analyses such as clustering/ classification/ regression.

In [ ]:
correlation_matrix = normalized_mightee.corr()
upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Find the index of feature columns with a correlation greater than 0.80
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.80)]
print(f"Columns to drop due to high correlation (>0.70): {to_drop}")

normalized_mightee.drop(columns=to_drop, inplace=True)

In [ ]:
normalized_mightee.columns

---

### Step 8: Data Visualisation

I'll leave the exploration of all 38 features to you, but for demonstration purposes, I will focus on just three:

- `MASS_lephare`
- `Rhat_SPIRE_500`
- `n_eff_SPIRE_250`
- `COS_best_z_v5`

These features will allow us to illustrate the key concepts and techniques without becoming overwhelmed by the full dimensionality of the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Select only the columns you want to plot from your normalized features
columns_to_plot = ["qir", "MASS_lephare", 'Rhat_SPIRE_500', 'n_eff_SPIRE_250', "COS_best_z_v5"]

# 2. Join the features and target cleanly using their matching index
# This discards rows that are not present in BOTH datasets automatically
plot_df = normalized_mightee[columns_to_plot].join(y, how="inner")

# 3. If your target 'y' Series didn't have a column name, pandas name it 'labels' 
# Let's force rename the target column to make sure seaborn can find it
if "labels" not in plot_df.columns:
    # If the series was named something else, rename the last column to 'labels'
    plot_df.rename(columns={plot_df.columns[-1]: "labels"}, inplace=True)

# 4. Generate the pairplot safely
print(f"Plotting pairplot with {len(plot_df)} aligned data points...")
sns.pairplot(plot_df, hue="labels")
plt.show()

----

### Well Done, Now lets Save the data.

In [ ]:
# Before moving on, let's save our processed data as Pandas DataFrames so that we can easily load them in the next notebook for clustering.
import pandas as pd

# Save the normalized feature matrix
normalized_mightee.to_csv("data/normalized_mightee.csv", index=False)

# Save the labels/targets
pd.Series(y, name="target").to_csv("data/y.csv", index=False)